# Lab 1: The Plan-and-Execute Newsroom

Welcome to your first lab on AI Agents! You will implement a **Plan-and-Execute** agent that produces high-quality research reports — without any hardcoded pipeline.

## Core Concept: Plan-and-Execute

Traditional multi-agent systems hardcode a fixed sequence of steps (research → analyze → write). The **Plan-and-Execute** pattern separates two concerns:

1. **Plan:** Ask the LLM to decompose the query into an ordered list of sub-tasks, each assigned to a specialist.
2. **Execute:** Run each sub-task in order, feeding dependency results as context into the next step.

This means the agent decides *which* specialists to call and *in what order* — the orchestrator just drives the loop.

```
Query → Planner → [Step 1, Step 2, Step 3, ...] → Execute each → Synthesize
```

---

## Project: The Newsroom

You will build a newsroom agent with three specialists:

1. **Researcher** — Finds raw information and cites sources.
2. **Analyst** — Cross-references findings, rates confidence, flags gaps.
3. **Writer** — Synthesizes analysis into a polished executive briefing.

### Learning Objectives
- Write focused **system prompts** that constrain agent behaviour.
- Model a plan as **structured output** using Pydantic (`PlanStep`, `Plan`).
- Implement the **execute loop**: dispatch each step to the right specialist and pass dependency context forward.
- Store step results in a plain `dict` for lookup by dependency.

---

## Phase 0: Setup and Imports

Run the cell below to load your environment variables and configure the litellm model.

In [ ]:
import os
import asyncio
import logging
from typing import List

from litellm import acompletion
from pydantic import BaseModel, Field

import nest_asyncio
nest_asyncio.apply()

# Configuration
import os
from dotenv import load_dotenv

load_dotenv()
MODEL_NAME = os.getenv("MODEL_NAME", "gpt-4o")

import dotenv
dotenv.load_dotenv()


## Phase 1: Specialist Prompts

Fill in `SPECIALIST_PROMPTS` for each of the three specialists and implement `call_specialist()`.

**Key constraint:** Each specialist should refuse to do work outside its role.

In [ ]:
SPECIALIST_PROMPTS = {
    "researcher": """
    # --- YOUR CODE HERE ---
    # Write a system prompt for the Researcher specialist.
    # Hint: Tell it to ONLY research, cite sources, and return raw findings.
    # --- END YOUR CODE ---
    """,

    "analyst": """
    # --- YOUR CODE HERE ---
    # Write a system prompt for the Analyst specialist.
    # Hint: Tell it to evaluate info, flag contradictions, rate confidence.
    # --- END YOUR CODE ---
    """,

    "writer": """
    # --- YOUR CODE HERE ---
    # Write a system prompt for the Writer specialist.
    # Hint: Tell it to write clearly, preserve citations, be concise.
    # --- END YOUR CODE ---
    """,
}

async def call_specialist(specialist: str, task: str) -> str:
    """Invoke a named specialist with a task and return its response."""
    # TODO: Look up the system prompt for `specialist` from SPECIALIST_PROMPTS,
    #       call the LLM, and return the response content.
    # --- YOUR CODE HERE ---
    pass
    # --- END YOUR CODE ---

<details>
<summary><b>Click here to see the solution</b></summary>

```python
SPECIALIST_PROMPTS = {
    "researcher": """You are a Research Specialist. Your ONLY job is to find and retrieve relevant information. You do NOT analyze or write reports.

Your standards:
- Always cite your sources with URLs or document references.
- Retrieve from multiple sources to avoid single-source bias.
- If information is limited, acknowledge gaps rather than fabricating.
- Return raw findings organized by source. Do NOT editorialize or summarize.""",

    "analyst": """You are an Analysis Specialist. Your ONLY job is to evaluate information provided to you and extract insights.

Your standards:
- Cross-reference claims across multiple sources. Flag contradictions explicitly.
- Distinguish between facts, opinions, and speculation in the source material.
- Identify gaps: what important questions does the research NOT answer?
- Rate confidence: High (multiple corroborating sources), Medium (single credible source), Low (unverified).""",

    "writer": """You are a Writing Specialist. Your ONLY job is to take analyzed research and produce a clear, well-structured document.

Your standards:
- Structure with clear headings, topic sentences, and transitions.
- Preserve source citations from the research phase.
- Include confidence qualifiers from the analysis.
- Keep it concise. No filler.""",
}

async def call_specialist(specialist: str, task: str) -> str:
    """Invoke a named specialist with a task and return its response."""
    system_prompt = SPECIALIST_PROMPTS.get(specialist, SPECIALIST_PROMPTS["researcher"])
    response = await acompletion(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": task},
        ],
        max_tokens=1024,
    )
    return response.choices[0].message.content
```
</details>

## Phase 2: The Planner

Write `PLANNER_PROMPT` — it must tell the LLM to output steps with `step`, `task`, `specialist`, and `depends_on` fields.
Then implement `TaskPlanner.create_plan()` to call the LLM with `response_format=Plan` and return a list of step dicts.

In [ ]:
PLANNER_PROMPT = """
# --- YOUR CODE HERE ---
# Hint: Explain the three specialists, their roles, and ask the LLM to produce
# a minimal ordered plan with step numbers and dependency lists.
# --- END YOUR CODE ---
"""


# ── Data models ───────────────────────────────────────────────────────────────
# These Pydantic models tell the LLM exactly what JSON shape to return.

class PlanStep(BaseModel):
    step: int = Field(..., description="Sequence number of the step.")
    task: str = Field(..., description="The specific, actionable task to perform.")
    specialist: str = Field(..., description="Specialist to use: 'researcher', 'analyst', or 'writer'.")
    depends_on: List[int] = Field(default_factory=list, description="Step numbers this step depends on.")


class Plan(BaseModel):
    steps: List[PlanStep] = Field(..., description="Ordered list of steps to achieve the goal.")


# ── Planner ───────────────────────────────────────────────────────────────────

class TaskPlanner:
    """Decomposes a newsroom query into ordered specialist sub-tasks."""

    async def create_plan(self, query: str) -> List[dict]:
        # TODO: Call acompletion with response_format=Plan and PLANNER_PROMPT.
        #       Parse the JSON response into a Plan object and return its steps as dicts.
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---

<details>
<summary><b>Click here to see the solution</b></summary>

```python
PLANNER_PROMPT = """You are a precise task planner for a newsroom that produces research reports.

You have three specialists available:
- researcher: Finds and retrieves raw information, cites sources
- analyst: Evaluates findings, flags contradictions, rates confidence
- writer: Synthesizes analysis into a polished, well-structured report

Decompose the following query into a minimal ordered list of steps.
Each step must specify which specialist handles it and which prior steps it depends on.

Query: {query}

Rules:
- Use "researcher" to gather raw information (can be parallelised — no depends_on needed)
- Use "analyst" to evaluate research findings (depends on the researcher steps)
- Use "writer" only for the final synthesis (depends on the analyst step)"""


# ── Data models ───────────────────────────────────────────────────────────────

class PlanStep(BaseModel):
    step: int = Field(..., description="Sequence number of the step.")
    task: str = Field(..., description="The specific, actionable task to perform.")
    specialist: str = Field(..., description="Specialist to use: 'researcher', 'analyst', or 'writer'.")
    depends_on: List[int] = Field(default_factory=list, description="Step numbers this step depends on.")


class Plan(BaseModel):
    steps: List[PlanStep] = Field(..., description="Ordered list of steps to achieve the goal.")


# ── Planner ───────────────────────────────────────────────────────────────────

class TaskPlanner:
    """Decomposes a newsroom query into ordered specialist sub-tasks."""

    async def create_plan(self, query: str) -> List[dict]:
        response = await acompletion(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You are a precise task planner."},
                {"role": "user", "content": PLANNER_PROMPT.format(query=query)},
            ],
            response_format=Plan,
            timeout=60,
        )
        plan = Plan.model_validate_json(response.choices[0].message.content)
        return [step.model_dump() for step in plan.steps]
```
</details>

## Phase 3: The Execute Loop & Synthesize

Inside `NewsroomAgent.run()`:
- Call `_get_context()` to retrieve dependency results.
- Append context to the task string and call `call_specialist()`.
- Store each result in `results[step_num]`.

Implement `_synthesize()` — format all step results and call the LLM to produce the final cited report.

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)


# TODO: Write a planner prompt that instructs the LLM to decompose the query
#       into steps, each assigned to one of: "researcher", "analyst", "writer".
#       Include rules for when each specialist should be used and how to set depends_on.
PLANNER_PROMPT = """
# --- YOUR CODE HERE ---
# Hint: Explain the three specialists, their roles, and ask the LLM to produce
# a minimal ordered plan with step numbers and dependency lists.
# --- END YOUR CODE ---
"""


# ── Data models ───────────────────────────────────────────────────────────────
# These Pydantic models tell the LLM exactly what JSON shape to return.

class PlanStep(BaseModel):
    step: int = Field(..., description="Sequence number of the step.")
    task: str = Field(..., description="The specific, actionable task to perform.")
    specialist: str = Field(..., description="Specialist to use: 'researcher', 'analyst', or 'writer'.")
    depends_on: List[int] = Field(default_factory=list, description="Step numbers this step depends on.")


class Plan(BaseModel):
    steps: List[PlanStep] = Field(..., description="Ordered list of steps to achieve the goal.")


# ── Planner ───────────────────────────────────────────────────────────────────

class TaskPlanner:
    """Decomposes a newsroom query into ordered specialist sub-tasks."""

    async def create_plan(self, query: str) -> List[dict]:
        # TODO: Call acompletion with response_format=Plan and PLANNER_PROMPT.
        #       Parse the JSON response into a Plan object and return its steps as dicts.
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---


# ── Agent ─────────────────────────────────────────────────────────────────────

class NewsroomAgent:
    """
    A Plan-and-Execute newsroom agent.

    Phase 1 — Plan:    Decompose the query into specialist sub-tasks.
    Phase 2 — Execute: Run each sub-task with the right specialist,
                       passing dependency results as context.
    Phase 3 — Synthesize: Combine all step results into the final report.
    """

    def __init__(self):
        self.planner = TaskPlanner()

    def _get_context(self, step: dict, results: dict) -> str:
        """Build a context string from the results of dependency steps."""
        # TODO: Iterate over step["depends_on"], look up each dep in results,
        #       and join them into a formatted string.
        # --- YOUR CODE HERE ---
        return ""
        # --- END YOUR CODE ---

    async def run(self, query: str) -> dict:
        # Phase 1: Plan
        logger.info("Phase 1: Planning")
        plan = await self.planner.create_plan(query)

        for step in plan:
            logger.info(f"  Step {step['step']} [{step['specialist']}]: {step['task']}")

        # Phase 2: Execute
        logger.info("Phase 2: Executing")
        results = {}

        for step in plan:
            step_num = step["step"]
            specialist = step["specialist"]
            task = step["task"]

            # TODO: Get dependency context, build sub_task, call call_specialist,
            #       and store the result in results[step_num].
            # --- YOUR CODE HERE ---
            pass
            # --- END YOUR CODE ---

        # Phase 3: Synthesize
        logger.info("Phase 3: Synthesizing")
        final_answer = await self._synthesize(query, plan, results)

        return {
            "answer": final_answer,
            "metadata": {
                "plan": plan,
                "step_results": results,
            },
        }

    async def _synthesize(self, query: str, plan: list, results: dict) -> str:
        """Combine all step results into a final coherent answer."""
        # TODO: Format results_text as "Step N (task): answer" and call acompletion
        #       with a system prompt asking for a synthesized, cited final answer.
        # --- YOUR CODE HERE ---
        pass
        # --- END YOUR CODE ---

<details>
<summary><b>Click here to see the solution</b></summary>

```python
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)


PLANNER_PROMPT = """You are a precise task planner for a newsroom that produces research reports.

You have three specialists available:
- researcher: Finds and retrieves raw information, cites sources
- analyst: Evaluates findings, flags contradictions, rates confidence
- writer: Synthesizes analysis into a polished, well-structured report

Decompose the following query into a minimal ordered list of steps.
Each step must specify which specialist handles it and which prior steps it depends on.

Query: {query}

Rules:
- Use "researcher" to gather raw information (can be parallelised — no depends_on needed)
- Use "analyst" to evaluate research findings (depends on the researcher steps)
- Use "writer" only for the final synthesis (depends on the analyst step)"""


# ── Data models ───────────────────────────────────────────────────────────────

class PlanStep(BaseModel):
    step: int = Field(..., description="Sequence number of the step.")
    task: str = Field(..., description="The specific, actionable task to perform.")
    specialist: str = Field(..., description="Specialist to use: 'researcher', 'analyst', or 'writer'.")
    depends_on: List[int] = Field(default_factory=list, description="Step numbers this step depends on.")


class Plan(BaseModel):
    steps: List[PlanStep] = Field(..., description="Ordered list of steps to achieve the goal.")


# ── Planner ───────────────────────────────────────────────────────────────────

class TaskPlanner:
    """Decomposes a newsroom query into ordered specialist sub-tasks."""

    async def create_plan(self, query: str) -> List[dict]:
        response = await acompletion(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You are a precise task planner."},
                {"role": "user", "content": PLANNER_PROMPT.format(query=query)},
            ],
            response_format=Plan,
            timeout=60,
        )
        plan = Plan.model_validate_json(response.choices[0].message.content)
        return [step.model_dump() for step in plan.steps]


# ── Agent ─────────────────────────────────────────────────────────────────────

class NewsroomAgent:
    """
    A Plan-and-Execute newsroom agent.

    Phase 1 — Plan:    Decompose the query into specialist sub-tasks.
    Phase 2 — Execute: Run each sub-task with the right specialist,
                       passing dependency results as context.
    Phase 3 — Synthesize: Combine all step results into the final report.
    """

    def __init__(self):
        self.planner = TaskPlanner()

    def _get_context(self, step: dict, results: dict) -> str:
        """Build a context string from the results of dependency steps."""
        parts = []
        for dep in step.get("depends_on", []):
            if dep in results:
                parts.append(f"[Result from Step {dep}]:\n{results[dep]}")
        return "\n\n".join(parts)

    async def run(self, query: str) -> dict:
        # Phase 1: Plan
        logger.info("Phase 1: Planning")
        plan = await self.planner.create_plan(query)

        for step in plan:
            logger.info(f"  Step {step['step']} [{step['specialist']}]: {step['task']}")

        # Phase 2: Execute
        logger.info("Phase 2: Executing")
        results = {}

        for step in plan:
            step_num = step["step"]
            specialist = step["specialist"]
            task = step["task"]

            context = self._get_context(step, results)
            sub_task = task if not context else f"{task}\n\nContext from previous steps:\n{context}"

            logger.info(f"  Executing Step {step_num} [{specialist}]")
            results[step_num] = call_specialist(specialist, sub_task)

        # Phase 3: Synthesize
        logger.info("Phase 3: Synthesizing")
        final_answer = await self._synthesize(query, plan, results)

        return {
            "answer": final_answer,
            "metadata": {
                "plan": plan,
                "step_results": results,
            },
        }

    async def _synthesize(self, query: str, plan: list, results: dict) -> str:
        """Combine all step results into a final coherent answer."""
        results_text = "\n\n".join(
            f"Step {num} ({next((p['task'] for p in plan if p['step'] == num), 'Unknown')}):\n{answer}"
            for num, answer in sorted(results.items())
        )
        response = await acompletion(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Synthesize the research results into a clear, well-structured answer. "
                        "Cite which step each piece of information came from."
                    ),
                },
                {
                    "role": "user",
                    "content": f"Original question: {query}\n\nResearch results:\n{results_text}",
                },
            ],
            timeout=60,
        )
        return response.choices[0].message.content
```
</details>

## Phase 4: Run the Agent

In [ ]:
agent = NewsroomAgent()
result = await agent.run("Compare the environmental policies of the EU and US")
print(f"\nFinal Report:\n{result['answer']}")